# Investor Tracking — pyecharts version

Same data, fresh start with **pyecharts** (Apache ECharts) for proper mobile responsiveness.

Run all → produces `index.html`. The file is automatically responsive: ECharts handles touch gestures, auto-resize, and scaling without margin gymnastics.

In [84]:
%pip install -q openpyxl pyecharts

Note: you may need to restart the kernel to use updated packages.


In [85]:
import re
from datetime import datetime, time
import numpy as np
import pandas as pd

from pyecharts.charts import Bar, Scatter, Grid
from pyecharts import options as opts
from pyecharts.commons.utils import JsCode

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Upload TWO Cash Flow Portal Excel exports — the latest and the one before')
    uploaded = files.upload()
    _all = sorted(uploaded.keys())
    SRC      = _all[-1]
    SRC_PREV = _all[-2] if len(_all) >= 2 else None
else:
    import glob as _glob
    _candidates = sorted(_glob.glob('Cash_Flow_Portal_Investments_*.xlsx'))
    SRC      = _candidates[-1] if _candidates else 'Cash_Flow_Portal_Investments_05_20_2026.xlsx'
    SRC_PREV = _candidates[-2] if len(_candidates) >= 2 else None

# ---- Override: pin the previous snapshot to a specific date ----
# Set PREV_DATE_OVERRIDE to a date string in MM_DD_YYYY format (e.g. '05_15_2026')
# to compare changes.html against that file. Set to None for auto (second-most-recent).
PREV_DATE_OVERRIDE = '05_22_2026'   # e.g. '05_15_2026'

if PREV_DATE_OVERRIDE:
    import pathlib as _pl
    _override = f'Cash_Flow_Portal_Investments_{PREV_DATE_OVERRIDE}.xlsx'
    if _pl.Path(_override).exists():
        SRC_PREV = _override
    else:
        print(f'WARN: override file {_override} not found — falling back to auto.')

print(f'Current : {SRC}')
print(f'Previous: {SRC_PREV}')

# TODAY = datetime(2026, 5, 5)
TODAY = datetime.combine(datetime.today(), time.min)
# print(TODAY)

WARN: override file Cash_Flow_Portal_Investments_05_22_2026.xlsx not found — falling back to auto.
Current : Cash_Flow_Portal_Investments_06_01_2026.xlsx
Previous: Cash_Flow_Portal_Investments_05_30_2026x.xlsx


## 1. Load + clean data (same as before)

In [86]:
STAGES = [
    'Investment started', 'Committed', 'Document signing started',
    '1 of 3 signed', 'Signed', 'Funding instructions sent',
    'Funds fully received', 'Counter-signed',
]

df = pd.read_excel(SRC, sheet_name='Investors (profile)')
df['Investor'] = (df['First name'].fillna('').astype(str).str.strip() + ' ' +
                  df['Last name'].fillna('').astype(str).str.strip()).str.strip()
df.loc[df['Investor'] == '', 'Investor'] = df['Profile name']
df['Invested amount'] = pd.to_numeric(df['Invested amount'], errors='coerce').fillna(0)
df['Funded amount']   = pd.to_numeric(df['Funded amount'], errors='coerce').fillna(0)
df['Gap'] = df['Invested amount'] - df['Funded amount']
df['IsCanceled'] = df['Status'].astype(str).str.lower().str.startswith('canceled')
df['Funded'] = (df['Funded amount'] >= df['Invested amount']) & (df['Invested amount'] > 0)
df['FundedLabel'] = np.where(df['IsCanceled'], 'Canceled',
                             np.where(df['Funded'], 'Funded', 'Unfunded'))
df['Date placed'] = pd.to_datetime(df['Date placed'], errors='coerce')
df['Days waiting'] = (TODAY - df['Date placed']).dt.days.fillna(0).astype(int)

active = df[~df['IsCanceled']]
summary = (active.groupby('Selected sponsors')
             .agg(investors=('Investor','count'),
                  committed=('Invested amount','sum'),
                  funded=('Funded amount','sum'),
                  gap=('Gap','sum'),
                  to_call=('Funded', lambda s: (~s).sum()))
             .sort_values('gap', ascending=False)
             .reset_index())

# Pizza-tracker stages — partial signing maps to 'Signed'
PIZZA_STAGES = [
    'Investment started', 'Committed', 'Document signing started',
    'Signed', 'Funding instructions sent', 'Funds fully received',
    'Counter-signed',
]
STAGE_NAMES_SHORT = ['Started', 'Committed', 'Doc opened', 'Signed', 'Fund instr', 'Recv', 'C-signed']
N_PIZZA = len(PIZZA_STAGES)

def map_pizza(status):
    if status in PIZZA_STAGES:
        return PIZZA_STAGES.index(status), False, None
    m = re.match(r'(\d+)\s*of\s*(\d+)\s*signed', str(status), re.IGNORECASE)
    if m:
        return PIZZA_STAGES.index('Signed'), True, f'{m.group(1)}/{m.group(2)}'
    return 0, False, None

df_bar = df[(df['Status'] != 'Counter-signed') & (~df['IsCanceled'])].copy()
_pi = df_bar['Status'].apply(map_pizza)
df_bar['curr_x']        = _pi.apply(lambda t: t[0])
df_bar['is_partial']    = _pi.apply(lambda t: t[1])
df_bar['partial_frac']  = _pi.apply(lambda t: t[2])
df_bar['EffectiveLabel'] = df_bar.apply(
    lambda r: 'Partial signing' if r['is_partial'] else r['FundedLabel'], axis=1)

# Sort by sponsor (priority) then days desc
sponsor_order = summary['Selected sponsors'].tolist()
df_bar['sp_rank'] = df_bar['Selected sponsors'].map({s:i for i,s in enumerate(sponsor_order)})
df_bar = df_bar.sort_values(['sp_rank', 'Days waiting'], ascending=[True, True]).reset_index(drop=True)

print(f'{len(df_bar)} active investors across {df_bar["Selected sponsors"].nunique()} sponsors')
df_bar.head()

35 active investors across 12 sponsors


,Unique investment id,Unique profile id,Profile name,Profile type,First name,Last name,Deal name,Offering name,Entity name,Email address,...,Gap,IsCanceled,Funded,FundedLabel,Days waiting,curr_x,is_partial,partial_frac,EffectiveLabel,sp_rank
0,5U1Y1VC8BU,1JADR0FWWS,Deepanshu Jain,individual,Deepanshu,Jain,Cambridge River Oaks Apartments,Cambridge River Oaks Apartments,-,Redacted,...,100000,False,False,Unfunded,59,4,False,NaN,Unfunded,0
1,AHFRHO6WU6,30XCB9OIGZ,Nathan Mullins,individual,Nathan,Mullins,Cambridge River Oaks Apartments,Cambridge River Oaks Apartments,-,Redacted,...,75000,False,False,Unfunded,68,4,False,NaN,Unfunded,0
2,CGWS3C2P07,6NJ1FOZFCE,James Lennon,individual,James,Lennon,Cambridge River Oaks Apartments,Cambridge River Oaks Apartments,-,Redacted,...,340000,False,False,Unfunded,82,4,False,NaN,Unfunded,0
3,DNQMRUTE1M,AQCQJO5GXZ,James I Boyd III,entity,James,Boyd,Cambridge River Oaks Apartments,Cambridge River Oaks Apartments,James I Boyd III,Redacted,...,0,False,False,Unfunded,6,0,False,NaN,Unfunded,1
4,CSHGL606P4,ESF7IMRYVU,"Selket Skorpios, LLC",entity,ABHISHEK,AGRAWAL,Cambridge River Oaks Apartments,Cambridge River Oaks Apartments,"Selket Skorpios, LLC",Redacted,...,0,False,True,Funded,10,5,False,NaN,Funded,1


## New investors since previous snapshot

Compares the current file against the previous snapshot (next-most-recent in this folder). New = `Unique investment id` present in current but absent in previous.

In [87]:
import html, pathlib
from datetime import datetime as _dt
import pandas as pd

def _fmt_date(v, fmt='%m/%d/%Y'):
    """Format a date value to MM/DD/YYYY (or given fmt). Return None if missing."""
    if v is None or v == '-' or (isinstance(v, float) and pd.isna(v)):
        return None
    if isinstance(v, str):
        s = v.strip()
        if not s or s == '-':
            return None
        for p in ('%m/%d/%Y %H:%M', '%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%m/%d/%Y'):
            try:
                return _dt.strptime(s, p).strftime(fmt)
            except ValueError:
                continue
        return s
    try:
        return v.strftime(fmt)
    except Exception:
        return str(v)

def _funded_date_cell(received, sent):
    """Build HTML for the 'Funded date' cell.

    If Received date is populated, show it (no superscript — the receipt date
    is the authoritative funding event).
    Only when Received date is missing do we fall back to a 'initiated MM/DD'
    superscript so the row still carries some context.
    """
    recv = _fmt_date(received)
    if recv:
        return html.escape(recv)
    sent_short = _fmt_date(sent, '%m/%d')
    if sent_short:
        return f'- <sup class="sent-note">initiated funding on {html.escape(sent_short)}</sup>'
    return '-' 

new_investors = pd.DataFrame()
newly_funded = pd.DataFrame()

if SRC_PREV:
    prev_df = pd.read_excel(SRC_PREV, sheet_name='Investors (profile)')
    prev_df['Invested amount'] = pd.to_numeric(prev_df['Invested amount'], errors='coerce').fillna(0)
    prev_df['Funded amount']   = pd.to_numeric(prev_df['Funded amount'], errors='coerce').fillna(0)
    prev_df['was_funded'] = (prev_df['Funded amount'] >= prev_df['Invested amount']) & (prev_df['Invested amount'] > 0)

    prev_ids = set(prev_df['Unique investment id'].dropna().astype(str))
    curr_ids = set(df['Unique investment id'].dropna().astype(str))
    new_ids = curr_ids - prev_ids

    if new_ids:
        new_investors = df[df['Unique investment id'].astype(str).isin(new_ids)][[
            'Investor', 'Selected sponsors', 'Status',
            'Invested amount', 'Funded amount', 'Days waiting',
        ]].copy()
        new_investors.columns = ['Investor', 'Sponsor', 'Stage', 'Invested', 'Funded', 'Days waiting']
        new_investors = new_investors.sort_values(['Sponsor','Invested'], ascending=[True, False]).reset_index(drop=True)

    # Newly funded: funded in current, NOT funded in previous (excludes brand-new investors who funded same day)
    prev_funded = set(prev_df.loc[prev_df['was_funded'], 'Unique investment id'].dropna().astype(str))
    curr_funded = set(df.loc[df['Funded'], 'Unique investment id'].dropna().astype(str))
    newly_funded_ids = curr_funded - prev_funded
    if newly_funded_ids:
        nf = df[df['Unique investment id'].astype(str).isin(newly_funded_ids)][[
            'Investor', 'Selected sponsors', 'Status',
            'Invested amount', 'Funded amount', 'Received date', 'Funds sent at',
        ]].copy()
        nf['Funded date'] = [
            _funded_date_cell(r, s) for r, s in zip(nf['Received date'], nf['Funds sent at'])
        ]
        newly_funded = nf[['Investor', 'Selected sponsors', 'Status',
                           'Invested amount', 'Funded amount', 'Funded date']].copy()
        newly_funded.columns = ['Investor', 'Sponsor', 'Stage', 'Invested', 'Funded', 'Funded date']
        newly_funded = newly_funded.sort_values(['Sponsor','Funded'], ascending=[True, False]).reset_index(drop=True)

prev_label = pathlib.Path(SRC_PREV).stem if SRC_PREV else 'N/A'
print(f'New investors since {prev_label}: {len(new_investors)}')
print(f'Newly funded since {prev_label}: {len(newly_funded)}')

New investors since Cash_Flow_Portal_Investments_05_30_2026x: 0
Newly funded since Cash_Flow_Portal_Investments_05_30_2026x: 2


## Separate changes report (changes.html)

Writes `changes.html` — a standalone page with two tables: **New investors** and **Newly funded** since the previous snapshot.

In [88]:
import html, pathlib
from datetime import datetime as _dt

def _table_html(df_, title, cols_money, raw_cols=None):
    raw_cols = raw_cols or set()
    if df_ is None or len(df_) == 0:
        return f'<section><h2>{html.escape(title)}</h2><p class="empty">None.</p></section>'
    headers = ''.join(
        f'<th class="num">{html.escape(c)}</th>' if c in cols_money else f'<th>{html.escape(c)}</th>'
        for c in df_.columns
    )
    rows = []
    for _, r in df_.iterrows():
        cells = []
        for c in df_.columns:
            v = r[c]
            if c in cols_money:
                cells.append(f'<td class="num">${float(v):,.0f}</td>' if pd.notna(v) else '<td class="num">$0</td>')
            elif c in raw_cols:
                cells.append(f'<td>{v}</td>')
            else:
                cells.append(f'<td>{html.escape(str(v))}</td>')
        rows.append('<tr>' + ''.join(cells) + '</tr>')

    # Total row across the numeric columns
    foot_cells = []
    for i, c in enumerate(df_.columns):
        if c in cols_money:
            total = pd.to_numeric(df_[c], errors='coerce').fillna(0).sum()
            foot_cells.append(f'<td class="num"><b>${total:,.0f}</b></td>')
        elif i == 0:
            foot_cells.append('<td><b>Total</b></td>')
        else:
            foot_cells.append('<td></td>')
    foot = '<tr class="total-row">' + ''.join(foot_cells) + '</tr>'

    return (
        f'<section><h2>{html.escape(title)} <span class="count">({len(df_)})</span></h2>'
        '<div class="table-wrap"><table>'
        f'<thead><tr>{headers}</tr></thead>'
        f'<tbody>{"".join(rows)}</tbody>'
        f'<tfoot>{foot}</tfoot>'
        '</table></div></section>'
    )

prev_label = pathlib.Path(SRC_PREV).stem.replace('Cash_Flow_Portal_Investments_', '').replace('_','/') if SRC_PREV else 'N/A'
now_str = _dt.now().astimezone().strftime('%B %d, %Y at %-I:%M %p %Z')

new_html  = _table_html(new_investors, f'New investors since {prev_label}', {'Invested', 'Funded'})
fund_html = _table_html(newly_funded,  f'Newly funded since {prev_label}',  {'Invested', 'Funded'}, raw_cols={'Funded date'})

changes_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Investor Tracking — Changes</title>
<style>
  html, body {{ margin:0; padding:0; background:#fafafa;
                font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;
                color:#222; }}
  .container {{ max-width: 1000px; margin: 0 auto; padding: 16px; }}
  h1 {{ font-size: 18px; margin: 6px 0 4px; color:#0d2240; text-align:center; }}
  .as-of {{ text-align:center; color:#666; font-size:12px; font-style:italic; margin-bottom: 18px; }}
  section {{ background: white; border: 1px solid #e0e0e0; border-radius: 8px;
             padding: 12px 14px; margin-bottom: 14px;
             box-shadow: 0 1px 3px rgba(0,0,0,0.04); }}
  h2 {{ margin: 0 0 8px; font-size: 14px; color: #0d2240; font-weight: 700; }}
  h2 .count {{ color: #a00000; font-weight: 600; font-size: 13px; margin-left: 6px; }}
  .table-wrap {{ overflow-x: auto; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 12px; }}
  th {{ text-align: left; padding: 6px 8px; color: #666; font-weight: 600;
       border-bottom: 1px solid #e0e0e0; font-size: 11px;
       text-transform: uppercase; letter-spacing: 0.3px; white-space: nowrap; }}
  th.num, td.num {{ text-align: right; font-variant-numeric: tabular-nums; }}
  td {{ padding: 6px 8px; border-bottom: 1px solid #f3f3f3; }}
  p.empty {{ margin: 0; color: #888; font-style: italic; font-size: 12px; }}
  sup.sent-note {{ font-size: 9px; color: #888; font-weight: 500;
                   margin-left: 4px; white-space: nowrap; vertical-align: super; }}
  @media (max-width: 640px) {{ th, td {{ padding: 5px 6px; font-size: 11px; }} }}
</style>
</head>
<body>
  <div class="container">
    <h1>Investor Tracking — Changes</h1>
    <div class="as-of">As of {now_str}</div>
    {new_html}
    {fund_html}
  </div>
</body>
</html>"""

pathlib.Path('changes.html').write_text(changes_html, encoding='utf-8')
print(f'Saved changes.html  (new={len(new_investors)}, newly funded={len(newly_funded)})')

Saved changes.html  (new=0, newly funded=2)


## 2. Build the chart with pyecharts

Two side-by-side grids:
- **Left** — pizza tracker: 7 stage dots per investor (✓ for completed, colored ring for current, empty for future)
- **Right** — horizontal bar of days waiting, colored by funded status

Mobile responsiveness is built into Apache ECharts (auto-resize, touch, pinch-zoom).

In [89]:
import json, html

TARGETS = {
    'Mattanza Capital Partners':        1_500_000,
    'K&G Capital':                        700_000,
    'Fast Fire Capital Llc':            1_500_000,
    'James Lennon':                       500_000,
    'Rashmi And Prem':                    250_000,
    'Manka Capital, Chirag Chaudhari':  1_500_000,
    'Prime Real Ventures':                250_000,
    'Alara Equity Llc':                   250_000,
    'Bh Group':                           250_000,
    'Sai Capital Llc':                  1_500_000,
    'Prysm2025':                          500_000,
    'New Horizon Capital Group':          500_000,
    'Optic Point':                        250_000,
}

def _fmt_target_label(k):
    if k >= 1000 and k % 1000 == 0: return f"${k//1000}M"
    if k >= 1000: return f"${k/1000:.1f}M"
    return f"${k}k"

def target_section_html(sp, committed, funded, gap, follow):
    target = TARGETS.get(sp, 0)
    if target <= 0: return ''
    inv_k = int(committed) // 1000
    fnd_k = int(funded) // 1000
    gap_k = int(gap) // 1000
    tgt_k = int(target) // 1000
    fpct = fnd_k / tgt_k * 100
    cpct = inv_k / tgt_k * 100
    hue = min(round(fpct * 1.2), 120)
    sat = round(60 + (120 - hue) * 0.13)
    light = round(40 + (120 - hue) / 12)
    fedge = ' edge-right' if fpct >= 100 else ''
    cedge = ' edge-right' if cpct >= 100 else ''
    fpos = 100 if fpct >= 100 else round(fpct, 1)
    cpos = 100 if cpct >= 100 else round(cpct, 1)
    follow = int(follow)
    investor_word = "investor" if follow == 1 else "investors"
    lines = []
    if fnd_k >= tgt_k:
        pct = round(fnd_k / tgt_k * 100)
        lines.append(f'<div class="reached">&#10003; Target reached &middot; ${fnd_k:,}k funded ({pct}%)</div>')
    if gap_k > 0 and follow > 0:
        lines.append(f'<div>Collect <b class="gap">${gap_k:,}k</b> from <b>{follow} committed {investor_word}</b></div>')
    if inv_k < tgt_k:
        new_k = tgt_k - inv_k
        lines.append(f'<div>Raise <b>${new_k:,}k</b> from new investors</div>')
    status_html = "\n          ".join(lines)
    target_lbl = _fmt_target_label(tgt_k)
    return f'''      <div class="target-section" style="--start-color: hsl({hue}, {sat}%, {light}%)">
        <div class="target-amounts">
          <span class="target-amt">Target {target_lbl}</span>
        </div>
        <div class="target-bar-wrap">
          <div class="target-bar"></div>
          <div class="target-marker funded{fedge}" style="left:{fpos}%">${fnd_k:,}k funded</div>
          <div class="target-marker committed{cedge}" style="left:{cpos}%">${inv_k:,}k committed</div>
        </div>
        <div class="target-status">
          {status_html}
        </div>
      </div>
'''

COLOR = {
    'Funded':          '#2ca02c',
    'Unfunded':        '#d62728',
    'Partial signing': '#ff8c00',
    'Canceled':        '#9e9e9e',
}
DONE_COLOR    = '#1a6b1a'

# Sponsor pastel palette (same family as the Plotly version)
SWIM_PALETTE = [
    '#e8eded','#ede8e3','#e8eaed','#ede8eb','#eaedeb','#edebe6',
    '#ebe9e6','#ebebeb','#ede5e5','#e5e8ed','#e8ede5','#ebe5ed','#ede9e3',
]

# Make duplicate investor names unique per row
_seen, _unique = {}, []
for _name in df_bar['Investor']:
    _seen[_name] = _seen.get(_name, 0) + 1
    _unique.append(_name if _seen[_name] == 1 else f'{_name} #{_seen[_name]}')
df_bar = df_bar.copy()
df_bar['UniqueLabel'] = _unique

# Max days across active investors → drives bar widths
MAX_DAYS = max(int(df_bar['Days waiting'].max()), 1)

def stage_dots_html(curr_x, fund_label, partial_frac):
    """Render the 7 stage dots for one investor row."""
    parts = ['<div class="track">']
    for st in range(N_PIZZA):
        if st > 0:
            parts.append('<div class="line"></div>')
        if st < curr_x:
            parts.append('<div class="dot done">&#10003;</div>')
        elif st == curr_x:
            color = COLOR.get(fund_label, '#888')
            inner = partial_frac if isinstance(partial_frac, str) else ''
            parts.append(f'<div class="dot current" style="background:{color};border-color:{color}">{html.escape(inner)}</div>')
        else:
            parts.append('<div class="dot future"></div>')
    parts.append('</div>')
    return ''.join(parts)

def investor_row_html(r):
    label_class = {
        'Funded': 'funded', 'Unfunded': 'unfunded',
        'Partial signing': 'partial', 'Canceled': 'canceled'
    }.get(r['EffectiveLabel'], 'unfunded')
    days = int(r['Days waiting'])
    pct = min(100, days * 100 / MAX_DAYS)
    name = html.escape(str(r['UniqueLabel']))
    inv = r['Invested amount']; fnd = r['Funded amount']
    if 0 < fnd < inv:
        amt = f"${fnd/1000:,.0f}k / ${inv/1000:,.0f}k"
    else:
        amt = f"${inv/1000:,.0f}k"
    dots = stage_dots_html(int(r['curr_x']), r['EffectiveLabel'], r['partial_frac'])
    return f'''<div class="row {label_class}" title="{html.escape(str(r['Status']))}">
        <div class="name">{name}<span class="amt">{amt}</span></div>
        {dots}
        <div class="daybar-wrap">
          <div class="daybar-track"><div class="daybar" style="width:{pct:.1f}%"></div></div>
          <span class="days-label">{days}d</span>
        </div>
      </div>'''

# Build per-sponsor sections
sections_html = []
for sp in sponsor_order:
    grp = df_bar[df_bar['Selected sponsors'] == sp]
    # Hide fully-funded investors from the report (they've already arrived; sponsor stats stay)
    grp = grp[grp['EffectiveLabel'] != 'Funded']
    if grp.empty: continue
    srow = summary[summary['Selected sponsors'] == sp].iloc[0]
    sp_idx = sponsor_order.index(sp) % len(SWIM_PALETTE)
    swim = SWIM_PALETTE[sp_idx]
    rows_html = ''.join(investor_row_html(r) for _, r in grp.iterrows())
    sections_html.append(f'''
    <section class="sponsor" style="--lane-color:{swim}">
      <header>
        <span class="sp-name">{html.escape(str(sp))}</span>
        <span class="sp-stats"><b class="gap">${srow['gap']/1000:,.0f}k gap</b> · {int(srow['to_call'])} to follow up</span>
      </header>
{target_section_html(sp, srow['committed'], srow['funded'], srow['gap'], srow['to_call'])}      <div class="stage-header">
        <div class="name-col"></div>
        <div class="track-header">''' +
        ''.join(f'<span class="st-label">{html.escape(s)}</span><span class="st-num">{i+1}</span>' + ('<span class="st-spacer"></span>' if i < N_PIZZA-1 else '') for i, s in enumerate(STAGE_NAMES_SHORT))
        + '''</div>
        <div class="daybar-header">Days waiting</div>
      </div>
      {rows_html}
    </section>'''.replace('{rows_html}', rows_html))

from datetime import datetime as _dt
_now_str = _dt.now().astimezone().strftime('%B %d, %Y at %-I:%M %p %Z')

HTML_TEMPLATE = '''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Investor Tracking</title>
<style>
  :root { --done-color:#1a6b1a; --unfunded:#d62728; --funded:#2ca02c; --partial:#ff8c00; }
  html, body { margin:0; padding:0; background:#fafafa;
               font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;
               color:#222; }
  .container { max-width: 1100px; margin: 0 auto; padding: 12px; }
  h1 { font-size: 18px; margin: 6px 0 18px; color:#0d2240; text-align:center; }

  .as-of { text-align: center; font-size: 12px; color: #666;
           margin: -10px 0 6px; font-style: italic; }
  .report-note { text-align: center; font-size: 11px; color: #888;
                 margin: 0 auto 16px; max-width: 720px; padding: 0 12px;
                 font-style: italic; line-height: 1.4; }

  .sponsor { background: var(--lane-color); border-radius: 8px;
             margin-bottom: 14px; overflow: hidden;
             box-shadow: 0 1px 3px rgba(0,0,0,0.05); }
  .sponsor header { padding: 10px 14px; background: rgba(0,0,0,0.06);
                    display: flex; flex-wrap: wrap; gap: 6px 14px;
                    align-items: baseline; }
  .sp-name { font-weight: 700; font-size: 14px; color: #0d2240; }
  .sp-stats { font-size: 12px; color: #444; }
  .sp-stats .gap { color: #a00000; }

  /* Header row showing stage labels (desktop only) */
  .stage-header { display: grid;
                  grid-template-columns: minmax(140px,1.6fr) minmax(160px,3fr) minmax(110px,2fr);
                  gap: 10px; align-items: center;
                  padding: 6px 14px 4px; font-size: 10px; color: #1f3a5f; font-weight: 600; }
  .stage-header .track-header { display: flex; align-items: center; }
  .st-label { flex: 0 0 auto; font-size: 9px; }
  .st-num   { display: none; }
  .st-spacer { flex: 1 1 0; }

  /* Investor row */
  .row { display: grid;
         grid-template-columns: minmax(140px,1.6fr) minmax(160px,3fr) minmax(110px,2fr);
         gap: 10px; align-items: center;
         padding: 6px 14px; background: rgba(255,255,255,0.55);
         border-top: 1px solid rgba(0,0,0,0.05); }
  .row .name { font-size: 12px; color:#1f1f1f; display:flex; flex-direction:column; }
  .row .name .amt { font-size: 11px; color:#666; }

  .track { display: flex; align-items: center; }
  .track .line { flex: 1 1 0; height: 2px; background: #cfcfcf; }
  .dot { width: 16px; height: 16px; border-radius: 50%;
         display:flex; align-items:center; justify-content:center;
         font-size: 9px; font-weight: 700; color: white;
         flex: 0 0 16px; box-sizing: border-box; }
  .dot.done   { background: var(--done-color); border: 1.5px solid var(--done-color); }
  .dot.future { background: white; border: 1.5px solid #b0b0b0; }
  .dot.current { color: white; border: 2px solid white;
                 box-shadow: 0 0 0 1.5px currentColor; }

  .daybar-wrap { display: flex; align-items: center; gap: 8px; }
  .daybar-track { flex: 1 1 0; height: 10px; background: #eee; border-radius: 5px; min-width: 50px; }
  .daybar { height: 100%; border-radius: 5px; background: var(--unfunded); transition: width 0.3s; }
  .row.funded  .daybar { background: var(--funded); }
  .row.partial .daybar { background: var(--partial); }
  .row.canceled .daybar { background: #999; }
  .days-label { flex: 0 0 auto; font-size: 11px; color: #333; font-weight: 600;
                min-width: 32px; text-align: right; }
  .daybar-header { font-size: 10px; color: #1f3a5f; font-weight: 600; }
  .d-icon, .d-num { display: inline; }
  .d-num { display: none; }
  /* Top legends */
  .top-legend { background: white; border: 1px solid #e0e0e0; border-radius: 6px;
                padding: 10px 14px; margin-bottom: 14px; font-size: 12px;
                position: sticky; top: 0; z-index: 10; }
  .color-key { display: flex; flex-wrap: wrap; gap: 6px 16px; align-items: center; }
  .color-key .k { display: inline-flex; align-items: center; gap: 5px; }
  .color-key .chip { width: 14px; height: 14px; border-radius: 50%;
                     border: 1.5px solid white; box-shadow: 0 0 0 1px rgba(0,0,0,0.1);
                     display: inline-flex; align-items: center; justify-content: center;
                     color: white; font-size: 9px; font-weight: 700; line-height: 1; }
  .stage-key { display: none; margin-top: 8px; padding-top: 8px;
               border-top: 1px solid #eee;
               flex-wrap: wrap; gap: 4px 12px; font-size: 11px; color: #444;
               justify-content: center; }
  .stage-key b { color: #0d2240; }

  /* Target progress bar (per sponsor) */
  .target-section { padding: 8px 14px 10px; background: rgba(255,255,255,0.5);
                    border-top: 1px solid rgba(0,0,0,0.05); }
  .target-amounts { display: flex; justify-content: flex-end;
                    font-size: 11px; color: #0d2240; font-weight: 700;
                    margin-bottom: 6px; }
  .target-bar-wrap { position: relative; height: 14px;
                     margin: 22px 6px 24px; }
  .target-bar { position: absolute; inset: 0; border-radius: 7px;
                background: linear-gradient(to right, var(--start-color, #d62728), #2ca02c);
                box-shadow: inset 0 1px 2px rgba(0,0,0,0.15); }
  .target-marker { position: absolute; transform: translateX(-50%);
                   font-size: 10px; font-weight: 700; white-space: nowrap;
                   line-height: 1.1; }
  .target-marker.funded { bottom: 100%; margin-bottom: 1px; color: #0d4f0d; }
  .target-marker.funded::after { content: "\\25BC"; display: block;
                                 text-align: center; font-size: 9px;
                                 line-height: 1; }
  .target-marker.committed { top: 100%; margin-top: 1px; color: #1f3a5f; }
  .target-marker.committed::before { content: "\\25B2"; display: block;
                                     text-align: center; font-size: 9px;
                                     line-height: 1; }
  .target-marker.edge-right { transform: translateX(-100%); }
  .target-marker.edge-right::after,
  .target-marker.edge-right::before { text-align: right; }
  .target-status { font-size: 11px; color: #555; text-align: center;
                   margin-top: 8px; display: flex; flex-direction: column;
                   gap: 3px; align-items: center; }
  .target-status > div { line-height: 1.4; }
  .target-status b { color: #0d2240; font-weight: 700; }
  .target-status b.gap { color: #a00000; }
  .target-status .reached { color: #1a6b1a; font-weight: 700; }

  /* Mobile breakpoint: stack the 3 columns vertically inside each row */
  @media (max-width: 640px) {
    .container { padding: 6px; }
    /* Repurpose stage-header on mobile: show only the stage-number row above dots */
    .stage-header { display: block; padding: 4px 14px 0; grid-template-columns: none; }
    .stage-header .name-col, .stage-header .daybar-header { display: none; }
    .stage-header .track-header { display: flex; align-items: center; justify-content: space-between; }
    .st-label { display: none; }
    .st-num   { display: inline-block; font-size: 11px; font-weight: 700; color: #1f3a5f;
                width: 16px; text-align: center; flex: 0 0 16px; }
    .st-spacer { display: inline-block; flex: 1 1 0; }
    .row { grid-template-columns: 1fr; gap: 6px; padding: 8px 10px; }
    .row .name { flex-direction: row; align-items: baseline; gap: 8px; font-size: 13px; }
    .stage-key { display: flex; }
    .d-icon { display: none; }
    .d-num { display: inline; font-size: 9px; }
    .dot.future .d-num { color: #777; }
    .dot.done   .d-num { color: white; }
    .dot.current .d-num { color: white; }
    /* If current dot has both fraction AND number, show only number on mobile */
  }
</style>
</head>
<body>
  <div class="container">
    <h1>Investor Tracking</h1>
    <div class="as-of">As of AS_OF_PLACEHOLDER</div>
    <div class="report-note">Please note this is not a real-time report. There may be a delay between the date funds were sent and when the funds were received and updated. The report reflects the data in Cash Flow Portal as of AS_OF_PLACEHOLDER.</div>
    <div class="top-legend">
      <div class="color-key">
        <span class="k"><span class="chip" style="background:#d62728"></span>Unfunded</span>
        <span class="k"><span class="chip" style="background:#ff8c00"></span>Partial signing</span>
        <span class="k"><span class="chip chip-done" style="background:#1a6b1a">&#10003;</span>Completed stage</span>
      </div>
      <div class="stage-key">
        <span><b>1</b>=Started</span><span><b>2</b>=Committed</span><span><b>3</b>=Doc opened</span><span><b>4</b>=Signed</span><span><b>5</b>=Fund instr</span><span><b>6</b>=Recv</span><span><b>7</b>=C-signed</span>
      </div>
    </div>
    SECTIONS_PLACEHOLDER
  </div>
</body>
</html>
'''

html_out = (HTML_TEMPLATE
            .replace('AS_OF_PLACEHOLDER', _now_str)
            .replace('SECTIONS_PLACEHOLDER', '\n'.join(sections_html)))

import pathlib
pathlib.Path('index.html').write_text(html_out, encoding='utf-8')
print(f'Saved index.html  ({len(df_bar)} investors across {len(sections_html)} sponsor sections)')

Saved index.html  (35 investors across 12 sponsor sections)


## Totals by sponsor — CSV export

Writes `totals_by_sponsor.csv` (Sponsor, Committed, Funded, Gap) plus a total row. Open in Excel/Google Sheets.

In [90]:
import pandas as pd

TARGETS = {
    'Mattanza Capital Partners':        1_500_000,
    'K&G Capital':                        700_000,
    'Fast Fire Capital Llc':            1_500_000,
    'James Lennon':                       500_000,
    'Rashmi And Prem':                    250_000,
    'Manka Capital, Chirag Chaudhari':  1_500_000,
    'Prime Real Ventures':                250_000,
    'Alara Equity Llc':                   250_000,
    'Bh Group':                           250_000,
    'Sai Capital Llc':                  1_500_000,
    'Prysm2025':                          500_000,
    'New Horizon Capital Group':          500_000,
    'Optic Point':                        250_000,
}

totals = summary[['Selected sponsors', 'committed', 'funded']].copy()
totals.columns = ['Sponsor', 'Committed', 'Funded']
totals['Target'] = totals['Sponsor'].map(TARGETS).fillna(0).astype(int)
totals = totals[['Sponsor', 'Target', 'Committed', 'Funded']]

totals = pd.concat([totals, pd.DataFrame([{
        'Sponsor':   'Total',
        'Target':    totals['Target'].sum(),
        'Committed': totals['Committed'].sum(),
        'Funded':    totals['Funded'].sum(),
    }])], ignore_index=True)
    
totals.to_csv('totals_by_sponsor.csv', index=False)
print(f'Saved totals_by_sponsor.csv ({len(totals)-1} sponsors + Total row)')
totals

Saved totals_by_sponsor.csv (13 sponsors + Total row)


,Sponsor,Target,Committed,Funded
0,James Lennon,500000,615000,100000
1,Fast Fire Capital Llc,1500000,1280000,780000
2,Mattanza Capital Partners,1500000,1415000,945000
3,Alara Equity Llc,250000,675000,475000
4,Prysm2025,500000,275000,175000
5,Sai Capital Llc,1500000,525000,425000
6,"Manka Capital, Chirag Chaudhari",1500000,470000,375000
7,Optic Point,250000,235000,155000
8,New Horizon Capital Group,500000,150000,75000
9,Bh Group,250000,850000,800000


In [91]:
if IN_COLAB:
    files.download('index.html')
else:
    print('index.html ready in current directory.')

index.html ready in current directory.


In [92]:
8365000-5830000

2535000